# Projeto NPS Swift — Entrega Final Reprodutível

Este notebook consolida os pontos pedidos na entrega final: modelagem, inferência na base completa, proposta de NPS ajustado e análises de negócio.

O dashboard final fica no front React/TanStack Start, mas os cálculos principais foram concentrados em Python, principalmente no arquivo `api/nps_engine.py`.

## 1. Arquitetura escolhida

Foram usados modelos leves baseados em TF-IDF/classificadores clássicos como solução compatível com o tamanho da base, prazo de entrega, interpretabilidade e custo computacional. Para produção na Vercel, os modelos foram exportados para formato leve `.json.gz`, evitando dependências pesadas como `scikit-learn`, `numpy` e `scipy` no runtime serverless.

Arquivos principais:

- `scripts/treinar_modelo_sentimento.py`
- `scripts/treinar_modelo_categorizacao.py`
- `api/nps_engine.py`
- `api/lightweight_models.py`
- `api/models/modelo_sentimento_lite.json.gz`
- `api/models/modelo_categorizacao_lite.json.gz`

In [ ]:
from pathlib import Path
import csv, gzip, json

ROOT = Path.cwd()
if not (ROOT / "api").exists():
    # Caso o notebook seja aberto dentro de entrega/notebooks
    ROOT = ROOT.parents[1]

OUTPUTS = ROOT / "entrega" / "outputs"
MODELOS = ROOT / "entrega" / "modelos_salvos"
print("Raiz do projeto:", ROOT)
print("Outputs encontrados:", OUTPUTS.exists())
print("Modelos salvos encontrados:", MODELOS.exists())

## 2. Treinamento dos modelos

Os scripts de treino ficam versionados para reprodução. Eles aceitam diferentes nomes de colunas por aliases, tratam texto vazio/lixo e salvam modelos em `.joblib`.

Comandos de reprodução local:

```powershell
pip install -r requirements-modelos.txt
python scripts/treinar_modelo_sentimento.py --input "caminho/da/base.csv"
python scripts/treinar_modelo_categorizacao.py --input "caminho/da/base_anotada.csv"
```

A versão usada na Vercel é leve e fica em `api/models/*.json.gz`.

In [ ]:
for path in [
    ROOT / "scripts" / "treinar_modelo_sentimento.py",
    ROOT / "scripts" / "treinar_modelo_categorizacao.py",
    ROOT / "entrega" / "modelos_salvos" / "modelo_sentimento.joblib",
    ROOT / "entrega" / "modelos_salvos" / "modelo_categorizacao.joblib",
    ROOT / "api" / "models" / "modelo_sentimento_lite.json.gz",
    ROOT / "api" / "models" / "modelo_categorizacao_lite.json.gz",
]:
    print(path.relative_to(ROOT), "OK" if path.exists() else "NÃO ENCONTRADO")

## 3. Métricas registradas dos modelos

As métricas salvas ficam em `entrega/outputs/metricas_modelo_sentimento.csv` e `entrega/outputs/metricas_modelo_categorizacao.csv`. O modelo de sentimento também possui comparação com baseline/proxy pela classificação NPS.

In [ ]:
def read_rows(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

for fname in ["metricas_modelo_sentimento.csv", "metricas_modelo_categorizacao.csv"]:
    print("\n", fname)
    for row in read_rows(OUTPUTS / fname):
        print(row)

## 4. Inferência na base completa

A base completa inferida está em `entrega/outputs/base_inferida_completa.csv.gz`. Ela inclui os campos de sentimento, categoria, confiança, flags de baixa confiança e divergência nota × texto.

In [ ]:
base_path = OUTPUTS / "base_inferida_completa.csv.gz"
with gzip.open(base_path, "rt", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    rows_preview = []
    total = 0
    for row in reader:
        total += 1
        if len(rows_preview) < 3:
            rows_preview.append(row)

print("Linhas na base inferida:", total)
print("Colunas:", list(rows_preview[0].keys())[:20], "...")
print("Exemplo:")
rows_preview[0]

## 5. NPS tradicional e NPS ajustado

Fórmula proposta:

```text
NPS_ajustado = 0,7 × NPS_tradicional + 0,3 × Score_sentimento
```

Onde:

```text
NPS_tradicional = % promotores - % detratores
Score_sentimento = % positivos - % negativos
```

In [ ]:
resumo = read_rows(OUTPUTS / "resumo_geral.csv")[0]
for k in ["comentarios", "avaliacoes_ponderadas", "nps_tradicional", "score_sentimento", "nps_ajustado", "divergencia_%"]:
    print(k, "=", resumo.get(k))

## 6. Análise de impacto do NPS ajustado

A entrega pede análise de divergência entre nota e texto, impacto em lojas e casos extremos. Esses arquivos estão em:

- `impacto_nps_ajustado_lojas.csv`
- `casos_extremos_divergencia.csv`
- `qualidade_dados.csv`

In [ ]:
impacto = read_rows(OUTPUTS / "impacto_nps_ajustado_lojas.csv")
print("Top 5 lojas por diferença entre NPS ajustado e tradicional:")
for row in impacto[:5]:
    print(row["centronv2"], row["nps_tradicional"], row["nps_ajustado"], row["gap_ajustado_vs_tradicional"], row["mudou_faixa_executiva"])

print("\nCasos extremos de divergência:")
for row in read_rows(OUTPUTS / "casos_extremos_divergencia.csv")[:3]:
    print("-", row["classificacao_nps"], "x", row["sentimento_modelo"], "|", row["comentario"][:120])

## 7. Análises de negócio

A entrega pede comparativo por gestão, principais problemas/elogios, análise por loja e compilado executivo do sentimento. Os principais CSVs estão abaixo.

In [ ]:
for fname in ["comparativo_gestao.csv", "problemas_gerais.csv", "elogios_gerais.csv", "lojas_resumo.csv", "tendencia_mensal.csv"]:
    rows = read_rows(OUTPUTS / fname)
    print("\n", fname, "- linhas:", len(rows))
    for row in rows[:3]:
        print(row)

## 8. Geração dos outputs

Para regenerar os arquivos de entrega a partir da base inferida padrão:

```powershell
python scripts/gerar_outputs_entrega.py
```

Para regenerar a base padrão usada pelo front:

```powershell
python scripts/gerar_dados_nps.py
```

In [ ]:
print("Checklist final:")
for path in [
    "entrega/outputs/base_inferida_completa.csv.gz",
    "entrega/outputs/ANALISES_NEGOCIO.md",
    "entrega/outputs/impacto_nps_ajustado_lojas.csv",
    "entrega/modelos_salvos/README_MODELOS.md",
    "README.md",
    "vercel.json",
]:
    print(path, "OK" if (ROOT / path).exists() else "NÃO ENCONTRADO")